# 04 · Checkpoint Save, Load, and Generate

- **Objectives**: train a tiny model, save a DCP checkpoint, load it into a fresh model instance in the same process, then generate text from the loaded weights.
- **Requirements**: 1 GPU (CPU also works, just slower).
- **Runtime**: ~30 seconds (20 training steps + save + load + short generation).

## Setup

In [ ]:
import os
import shutil

import torch
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

CKPT_DIR = "/tmp/kf_nb04_ckpt"
if os.path.exists(CKPT_DIR):
    shutil.rmtree(CKPT_DIR)

## Build and train a tiny model (20 steps)

A quick training loop so parameters and optimizer state move off their random init. The task — next-token prediction on fresh random tokens each step — isn't learnable, so loss just hovers around `log(vocab_size) ≈ 5.54`. What we care about is that the model now holds non-initial state, so round-tripping it through a checkpoint is a real test.

In [ ]:
from kempnerforge.config.schema import ModelConfig, OptimizerConfig
from kempnerforge.model.transformer import Transformer
from kempnerforge.training.optimizer import build_optimizer

config = ModelConfig(
    dim=128,
    n_layers=4,
    n_heads=4,
    n_kv_heads=4,
    vocab_size=256,
    max_seq_len=64,
)

model = Transformer(config).to(device)
optimizer = build_optimizer(model, OptimizerConfig(name="adamw", lr=3e-3))

torch.manual_seed(0)
batch, seq_len = 4, 32
losses = []
for step in range(20):
    tokens = torch.randint(0, config.vocab_size, (batch, seq_len + 1), device=device)
    inp, tgt = tokens[:, :-1], tokens[:, 1:]
    logits = model(inp)
    loss = F.cross_entropy(logits.reshape(-1, config.vocab_size), tgt.reshape(-1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

print(f"Loss: {losses[0]:.3f} → {losses[-1]:.3f}")

## Save a checkpoint via `CheckpointManager`

The manager writes a DCP (Distributed Checkpoint) directory — works in single-process mode too, no distributed init required.

In [ ]:
from kempnerforge.checkpoint.manager import CheckpointManager
from kempnerforge.config.schema import CheckpointConfig

ckpt_config = CheckpointConfig(dir=CKPT_DIR, interval=99999, keep_last_n=2)
ckpt_mgr = CheckpointManager(ckpt_config, model, optimizer)

ckpt_mgr.save(step=20, tokens_seen=20 * batch * seq_len)
ckpt_mgr.wait()  # flush any async save

# Inspect what got written
for entry in sorted(os.listdir(CKPT_DIR)):
    path = os.path.join(CKPT_DIR, entry)
    print(f"  {entry}  ({'dir' if os.path.isdir(path) else 'file'})")

## Load into a fresh model

Instantiate a brand new `Transformer` (random weights) and verify the checkpoint load actually replaces the weights.

In [ ]:
fresh_model = Transformer(config).to(device)
fresh_optimizer = build_optimizer(fresh_model, OptimizerConfig(name="adamw", lr=3e-3))

# Baseline: random weights should differ from the trained model
trained_w = model.layers["0"].attention.q_proj.weight.detach().clone()
fresh_w_before = fresh_model.layers["0"].attention.q_proj.weight.detach().clone()
print(f"Before load — L2 diff: {(trained_w - fresh_w_before).norm().item():.4f}")

loader = CheckpointManager(ckpt_config, fresh_model, fresh_optimizer)
step, tokens_seen, extra = loader.load()
print(f"Loaded checkpoint: step={step}, tokens_seen={tokens_seen}, extra={extra}")

# After load — weights should match exactly
fresh_w_after = fresh_model.layers["0"].attention.q_proj.weight.detach().clone()
print(f"After load  — L2 diff: {(trained_w - fresh_w_after).norm().item():.4f}")

L2 difference should drop to ~0 after load — the fresh model now holds the trained weights.

## Generate from the loaded model

Quick sanity check: run `generate()` to produce tokens. The model wasn't trained on real text, so the output is gibberish — but we can verify the generation path works end-to-end.

In [ ]:
from kempnerforge.model.generate import generate

fresh_model.eval()
prompt = torch.randint(0, config.vocab_size, (1, 8), device=device)
with torch.no_grad():
    generated = generate(fresh_model, prompt, max_new_tokens=16, temperature=1.0, top_k=20)

print(f"Prompt tokens: {prompt[0].tolist()}")
print(f"Generated:     {generated[0].tolist()}")
print(f"New tokens:    {generated[0, prompt.size(1):].tolist()}")

## Cleanup

In [ ]:
shutil.rmtree(CKPT_DIR)
print(f"Removed {CKPT_DIR}")

## Notes

- DCP is **resharding-aware**: save with N GPUs (FSDP=N), resume with M (FSDP=M). Load figures out how to split/stitch weights.
- The `train_state.pt` file next to the DCP shards stores scheduler, dataloader, RNG, step counter, and any extras you passed via `ckpt_mgr.save(extra={...})`.
- For HuggingFace interop, use `scripts/convert_checkpoint.py` to convert a DCP checkpoint → HF format (and back).